# **Notebook01_Dataset Exploration**

### Import Library

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# **1. Load Dataset**

In [4]:
from pathlib import Path

# Lokasi folder data mentah
DATA_DIR = Path("../data/raw")

# Check if folder exist
print("Folder exist:", DATA_DIR.exists())

Folder exist: True


In [5]:
# Cari seluruh file CSV
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Jumlah file CSV: {len(csv_files)}")

# Baca semua file ke dalam dictionary
datasets = {}

for file in csv_files:
    table_name = file.stem
    datasets[table_name] = pd.read_csv(file)
    print(f"{table_name}: {datasets[table_name].shape}")

Jumlah file CSV: 9
olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_orders_dataset: (99441, 8)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


# **2. Datasets Exploration**

## **2.1 Tables Summary**

In [6]:
summary = []

for table_name, df in datasets.items():
    summary.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,table_name,rows,columns,missing_values,duplicate_rows
0,olist_customers_dataset,99441,5,0,0
1,olist_geolocation_dataset,1000163,5,0,261831
2,olist_order_items_dataset,112650,7,0,0
3,olist_order_payments_dataset,103886,5,0,0
4,olist_order_reviews_dataset,99224,7,145903,0
5,olist_orders_dataset,99441,8,4908,0
6,olist_products_dataset,32951,9,2448,0
7,olist_sellers_dataset,3095,4,0,0
8,product_category_name_translation,71,2,0,0


## **2.2 Columns and Data Types**

In [7]:
for table_name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"TABEL: {table_name}")
    print("=" * 60)
    print("Kolom:", list(df.columns))
    print("\nTipe data:")
    print(df.dtypes)


TABEL: olist_customers_dataset
Kolom: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Tipe data:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

TABEL: olist_geolocation_dataset
Kolom: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Tipe data:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

TABEL: olist_order_items_dataset
Kolom: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Tipe data:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping

In [8]:
date_columns = {
    "olist_orders_dataset": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "olist_order_items_dataset": [
        "shipping_limit_date"
    ],
    "olist_order_reviews_dataset": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

date_type_results = []

for table_name, columns in date_columns.items():
    df = datasets[table_name]

    for col in columns:
        date_type_results.append({
            "table": table_name,
            "column": col,
            "dtype": str(df[col].dtype)
        })

date_type_df = pd.DataFrame(date_type_results)

date_type_df

,table,column,dtype
0,olist_orders_dataset,order_purchase_timestamp,str
1,olist_orders_dataset,order_approved_at,str
2,olist_orders_dataset,order_delivered_carrier_date,str
3,olist_orders_dataset,order_delivered_customer_date,str
4,olist_orders_dataset,order_estimated_delivery_date,str
5,olist_order_items_dataset,shipping_limit_date,str
6,olist_order_reviews_dataset,review_creation_date,str
7,olist_order_reviews_dataset,review_answer_timestamp,str


## **2.3 Check for Missing Values**

In [9]:
for table_name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\nTABEL: {table_name}")

    if missing.empty:
        print("No missing values.")
    else:
        print(missing.sort_values(ascending=False))


TABEL: olist_customers_dataset
No missing values.

TABEL: olist_geolocation_dataset
No missing values.

TABEL: olist_order_items_dataset
No missing values.

TABEL: olist_order_payments_dataset
No missing values.

TABEL: olist_order_reviews_dataset
review_comment_title      87656
review_comment_message    58247
dtype: int64

TABEL: olist_orders_dataset
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

TABEL: olist_products_dataset
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

TABEL: olist_sellers_dataset
No missing values.

TABEL: product_category_name_translation
No missing values.


### **2.3.1 Missing Values in Orders**

In [10]:
#check dataset
datasets["olist_orders_dataset"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
orders = datasets["olist_orders_dataset"]

# Missing values berdasarkan status pesanan
missing_by_status = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_approved=("order_approved_at", lambda x: x.isna().sum()),
        missing_carrier=("order_delivered_carrier_date", lambda x: x.isna().sum()),
        missing_customer=("order_delivered_customer_date", lambda x: x.isna().sum())
    )
)

missing_by_status

,total_orders,missing_approved,missing_carrier,missing_customer
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


### **2.3.2 Missing Values in Products**

In [12]:
#check dataset
datasets["olist_products_dataset"].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [13]:
products = datasets["olist_products_dataset"]

missing_products = products.isna().sum()
missing_products = missing_products[missing_products > 0]

print("Missing values pada produk:")
print(missing_products)

print(
    "\nProducts without category:",
    products["product_category_name"].isna().sum()
)

print(
    "Products without name length:",
    products["product_name_lenght"].isna().sum()
)

print(
    "Products without description length:",
    products["product_description_lenght"].isna().sum()
)

print(
    "Products without photos quantity:",
    products["product_photos_qty"].isna().sum()
)

Missing values pada produk:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Products without category: 610
Products without name length: 610
Products without description length: 610
Products without photos quantity: 610


In [14]:
#check if missing values caused by unique products

product_missing_rows = products[
    products["product_category_name"].isna()
]

print("Products without category count:", len(product_missing_rows))
print("Unique products count:", product_missing_rows["product_id"].nunique())

Products without category count: 610
Unique products count: 610


## **2.4 Check for Duplicated Rows**

In [15]:
geo = datasets["olist_geolocation_dataset"]

# Duplikasi pada seluruh kolom
print("all columns duplicated:", geo.duplicated().sum())

# Jumlah kode pos unik
print("Unique zipcode:", geo["geolocation_zip_code_prefix"].nunique())


all columns duplicated: 261831
Unique zipcode: 19015


## **2.5 Primary Key Validation**

This section validates whether the expected primary keys are unique within each table.

In [16]:
# Expected primary keys
primary_keys = {
    "olist_customers_dataset": ["customer_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id"],
    "product_category_name_translation": ["product_category_name"]
}

pk_results = []

for table_name, pk_columns in primary_keys.items():
    df = datasets[table_name]

    duplicate_count = df.duplicated(
        subset=pk_columns
    ).sum()

    null_count = df[pk_columns].isna().any(axis=1).sum()

    pk_results.append({
        "table": table_name,
        "primary_key": ", ".join(pk_columns),
        "duplicate_keys": duplicate_count,
        "null_keys": null_count,
        "status": "PASS" if duplicate_count == 0 and null_count == 0 else "CHECK"
    })

pk_results_df = pd.DataFrame(pk_results)

pk_results_df

,table,primary_key,duplicate_keys,null_keys,status
0,olist_customers_dataset,customer_id,0,0,PASS
1,olist_orders_dataset,order_id,0,0,PASS
2,olist_products_dataset,product_id,0,0,PASS
3,olist_sellers_dataset,seller_id,0,0,PASS
4,olist_order_items_dataset,"order_id, order_item_id",0,0,PASS
5,olist_order_payments_dataset,"order_id, payment_sequential",0,0,PASS
6,olist_order_reviews_dataset,review_id,814,0,CHECK
7,product_category_name_translation,product_category_name,0,0,PASS


## **2.6 Foreign Key Validation**

This section checks whether foreign key values in child tables exist in their corresponding parent tables.

In [17]:
# Foreign key relationships
foreign_key_checks = [
    {
        "child_table": "olist_orders_dataset",
        "child_column": "customer_id",
        "parent_table": "olist_customers_dataset",
        "parent_column": "customer_id"
    },
    {
        "child_table": "olist_order_items_dataset",
        "child_column": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_column": "order_id"
    },
    {
        "child_table": "olist_order_items_dataset",
        "child_column": "product_id",
        "parent_table": "olist_products_dataset",
        "parent_column": "product_id"
    },
    {
        "child_table": "olist_order_items_dataset",
        "child_column": "seller_id",
        "parent_table": "olist_sellers_dataset",
        "parent_column": "seller_id"
    },
    {
        "child_table": "olist_order_payments_dataset",
        "child_column": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_column": "order_id"
    },
    {
        "child_table": "olist_order_reviews_dataset",
        "child_column": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_column": "order_id"
    }
]

fk_results = []

for check in foreign_key_checks:
    child_df = datasets[check["child_table"]]
    parent_df = datasets[check["parent_table"]]

    parent_values = set(
        parent_df[check["parent_column"]].dropna()
    )

    missing_fk = (
        ~child_df[check["child_column"]].isin(parent_values)
    ).sum()

    fk_results.append({
        "child_table": check["child_table"],
        "child_column": check["child_column"],
        "parent_table": check["parent_table"],
        "parent_column": check["parent_column"],
        "invalid_foreign_keys": missing_fk,
        "status": "PASS" if missing_fk == 0 else "CHECK"
    })

fk_results_df = pd.DataFrame(fk_results)

fk_results_df

,child_table,child_column,parent_table,parent_column,invalid_foreign_keys,status
0,olist_orders_dataset,customer_id,olist_customers_dataset,customer_id,0,PASS
1,olist_order_items_dataset,order_id,olist_orders_dataset,order_id,0,PASS
2,olist_order_items_dataset,product_id,olist_products_dataset,product_id,0,PASS
3,olist_order_items_dataset,seller_id,olist_sellers_dataset,seller_id,0,PASS
4,olist_order_payments_dataset,order_id,olist_orders_dataset,order_id,0,PASS
5,olist_order_reviews_dataset,order_id,olist_orders_dataset,order_id,0,PASS


## **2.7 Date Validation**

This section validates date data ranges, and chronological consistency between order timestamps.

In [18]:
#conversion to datetime
# ── Kolom datetime ──────────────────────────────────────────
datetime_map = {
    "olist_orders_dataset": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "olist_order_items_dataset": [
        "shipping_limit_date"
    ],
    "olist_order_reviews_dataset": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table_name, cols in datetime_map.items():
    df = datasets[table_name].copy()
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    datasets[table_name] = df
    print(f"{table_name}: datetime conversion done")

olist_orders_dataset: datetime conversion done
olist_order_items_dataset: datetime conversion done
olist_order_reviews_dataset: datetime conversion done


In [19]:
orders = datasets["olist_orders_dataset"].copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Pastikan semua kolom bertipe datetime
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Cari tanggal minimum dan maksimum dari seluruh kolom
date_range = pd.DataFrame({
    "min_date": orders[date_cols].min(),
    "max_date": orders[date_cols].max()
})

date_range

,min_date,max_date
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00


In [20]:
orders = datasets["olist_orders_dataset"]

invalid_purchase_approved = (
    orders["order_approved_at"].notna()
    &
    (orders["order_approved_at"] < orders["order_purchase_timestamp"])
)

invalid_approved_carrier = (
    orders["order_approved_at"].notna()
    &
    orders["order_delivered_carrier_date"].notna()
    &
    (orders["order_delivered_carrier_date"] < orders["order_approved_at"])
)

invalid_carrier_customer = (
    orders["order_delivered_carrier_date"].notna()
    &
    orders["order_delivered_customer_date"].notna()
    &
    (orders["order_delivered_customer_date"]
     < orders["order_delivered_carrier_date"])
)

print(
    "Approved before purchase:",
    invalid_purchase_approved.sum()
)

print(
    "Carrier before approved:",
    invalid_approved_carrier.sum()
)

print(
    "Customer delivery before carrier delivery:",
    invalid_carrier_customer.sum()
)

Approved before purchase: 0
Carrier before approved: 1359
Customer delivery before carrier delivery: 23


In [21]:
invalid_estimated = (
    orders["order_estimated_delivery_date"]
    < orders["order_purchase_timestamp"]
)

print(
    "Estimated delivery before purchase:",
    invalid_estimated.sum()
)

Estimated delivery before purchase: 0


In [22]:
reviews = datasets["olist_order_reviews_dataset"]

invalid_review_dates = (
    reviews["review_answer_timestamp"].notna()
    &
    reviews["review_creation_date"].notna()
    &
    (
        reviews["review_answer_timestamp"]
        < reviews["review_creation_date"]
    )
)

print(
    "Review answer before review creation:",
    invalid_review_dates.sum()
)

Review answer before review creation: 0


## **2.8 Analytical Grain**

The analytical grain defines what one row represents in each table. Understanding the grain prevents double counting when tables are joined.

### **2.8.1 Customers Table**

In [23]:
#customers table
customers = datasets["olist_customers_dataset"]

print(
    "Total rows:",
    len(customers)
)

print(
    "Unique customer_id:",
    customers["customer_id"].nunique()
)

print(
    "Duplicated customer_id:",
    customers["customer_id"].duplicated().sum()
)

print(
    "Unique customer_unique_id:",
    customers["customer_unique_id"].nunique()
)

print(
    "Duplicated customer_unique_id:",
    customers["customer_unique_id"].duplicated().sum()
)

Total rows: 99441
Unique customer_id: 99441
Duplicated customer_id: 0
Unique customer_unique_id: 96096
Duplicated customer_unique_id: 3345


### **2.8.2 Orders Table**

In [24]:
orders = datasets["olist_orders_dataset"]

print("Total rows:", len(orders))
print("Unique order_id:", orders["order_id"].nunique())
print(
    "Duplicate order_id:",
    orders["order_id"].duplicated().sum()
)

Total rows: 99441
Unique order_id: 99441
Duplicate order_id: 0


### **2.8.3 Order Items Table**

In [25]:
order_items = datasets["olist_order_items_dataset"]

print("Total rows:", len(order_items))
print("Unique order_id:", order_items["order_id"].nunique())
print("Duplicated order_id:", order_items["order_id"].duplicated().sum())
print("Unique order_item_id:", order_items["order_item_id"].nunique())
print("Duplicated order_item_id:", order_items["order_item_id"].duplicated().sum())
print(
    "Duplicated order_id + order_item_id:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Total rows: 112650
Unique order_id: 98666
Duplicated order_id: 13984
Unique order_item_id: 21
Duplicated order_item_id: 112629
Duplicated order_id + order_item_id: 0


### **2.8.4 Payments Table**

In [26]:
payments = datasets["olist_order_payments_dataset"]

print("Total rows:", len(payments))
print("Unique orders:", payments["order_id"].nunique())
print("Duplicated orders:", payments["order_id"].duplicated().sum())
print("Unique payment_sequentials:", payments["payment_sequential"].nunique())
print("Duplicated payment_sequentials:", payments["payment_sequential"].duplicated().sum())
print(
    "Duplicate order_id + payment_sequential:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

Total rows: 103886
Unique orders: 99440
Duplicated orders: 4446
Unique payment_sequentials: 29
Duplicated payment_sequentials: 103857
Duplicate order_id + payment_sequential: 0


### **2.8.5 Reviews Table**

In [27]:
reviews = datasets["olist_order_reviews_dataset"]

print("Total rows:", len(reviews))
print("Unique review_id:", reviews["review_id"].nunique())

print(
    "Duplicate review_id:",
    reviews["review_id"].duplicated().sum()
)

print("Unique order_id:", reviews["order_id"].nunique())

print(
    "Duplicate order_id:",
    reviews["order_id"].duplicated().sum()
)

print(
    "Duplicate order_id + review_id:",
    reviews.duplicated(
        subset=["order_id", "review_id"]
    ).sum()
)

Total rows: 99224
Unique review_id: 98410
Duplicate review_id: 814
Unique order_id: 98673
Duplicate order_id: 551
Duplicate order_id + review_id: 0


### **2.8.6 Products Table**

In [28]:
products = datasets["olist_products_dataset"]

print("Total rows:", len(products))
print("Unique product_id:", products["product_id"].nunique())
print(
    "Duplicate product_id:",
    products["product_id"].duplicated().sum()
)

Total rows: 32951
Unique product_id: 32951
Duplicate product_id: 0


### **2.8.7 Sellers Table**

In [29]:
sellers = datasets["olist_sellers_dataset"]

print("Total rows:", len(sellers))
print("Unique seller_id:", sellers["seller_id"].nunique())
print(
    "Duplicate seller_id:",
    sellers["seller_id"].duplicated().sum()
)

Total rows: 3095
Unique seller_id: 3095
Duplicate seller_id: 0


### **2.8.8 Geolocation Table**

In [31]:
geo = datasets["olist_geolocation_dataset"]

print("Total rows:", len(geo))
print(
    "Unique zip codes:",
    geo["geolocation_zip_code_prefix"].nunique()
)

print(
    "Duplicated zip codes:",
    geo["geolocation_zip_code_prefix"].duplicated().sum()
)

Total rows: 1000163
Unique zip codes: 19015
Duplicated zip codes: 981148


### **2.8.9 Product Category Name Translation Table**

In [32]:
translation = datasets[
    "product_category_name_translation"
]

print("Total rows:", len(translation))
print(
    "Unique Portuguese categories:",
    translation["product_category_name"].nunique()
)

print(
    "Duplicate category:",
    translation["product_category_name"].duplicated().sum()
)

Total rows: 71
Unique Portuguese categories: 71
Duplicate category: 0


# **3. Exploration Result**

## **3.1 Dataset Overview**
- Dataset name: Brazilian E-Commerce Public Dataset by Olist
- Source: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- Table count: 9
- Transaction period: 2016 - 2018
- Total orders: 99441

## **3.2. Data Quality**
- Tables with missing values: 3 (Reviews, orders, products)
- Tables with duplicates data: 1 (Geolocation)

## **3.3 Initial Findings**

### **Findings #1**
Several columns require data type adjustments.

### **Findings #2**
Missing values are found in the review table:
- review_comment_title: 87.656 missing values.
- review_comment_message: 58.247 missing values.

This findings are very reasonable, we can leave these fields empty since the ratings remain useful.

### **Findings #3**
Missing values are also found in orders table:
- order_delivered_customer_date: 2.965.
- order_delivered_carrier_date: 1.783.
- order_approved_at: 160.

The missing delivery-related dates should be interpreted together with order status. Orders that were canceled, unavailable, or still in earlier processing stages may legitimately have missing delivery timestamps.

Further validation is required before deciding whether these missing values represent data quality issues.

### **Findings #4**
Products table also have missing values:
- product_category_name, product_name_lenght product_description_lenght, product_photos_qty (610 missing values)
- product_weight_g, product_length_cm, product_height_cm, product_width_cm (2 missing values)


### **Findings #5**
- The geolocation dataset contains 1,000,163 rows and 19,015 unique zip code prefixes.
- A single zip code prefix can be associated with multiple geographic coordinate points.
- In addition, 261,831 rows are exact duplicates across all columns.

## **3.4 Next Investigation**

- Validate relationships between tables.
- Check primary key uniqueness.
- Check foreign key integrity.
- Validate date columns.
- Identify the analytical grain for InsightFlow.

### **3.4.1 Relationships Validation Results**

The relationship validation shows that all tested foreign key relationships are valid.

- All `customer_id` values in the orders table exist in the customers table.
- All `order_id` values in the order_items, payments, and reviews tables exist in the orders table.
- All `product_id` values in the order_items table exist in the products table.
- All `seller_id` values in the order_items table exist in the sellers table.

No invalid foreign key values were identified across the tested relationships. This indicates that the main transactional tables can be connected through their respective keys without unmatched records.

However, the different levels of granularity across tables must still be considered before performing joins to avoid row multiplication and double counting.

### **3.4.2 Date Validation Results**

The date validation shows that the main date columns were successfully converted to datetime format.

The order data covers purchase timestamps from September 4, 2016 to October 17, 2018. Estimated delivery dates extend to November 12, 2018, which is expected because estimated delivery dates may occur after the latest recorded purchase or actual delivery date.

No orders were found with an approval timestamp earlier than the purchase timestamp, and no estimated delivery dates occurred before the purchase timestamp.

However, 1,359 orders have carrier delivery dates earlier than their approval timestamps, while 23 orders have customer delivery dates earlier than their carrier delivery dates. These records require further investigation before being used for delivery-time analysis.

The review timestamps show no cases where a review answer timestamp occurred before the review creation date.

### **3.4.3 Primary Keys Validation Results**

Primary key validation shows that the expected keys are unique and non-null for most tables.

The customers, orders, products, sellers, order_items, payments, and product category translation tables all passed the primary key validation.

The reviews table requires further investigation because 814 duplicate `review_id` values were identified. However, the combination of `order_id` and `review_id` remains unique, indicating that the duplicated review IDs do not form duplicate records under the tested composite key.

Therefore, `review_id` should not be treated as a fully reliable unique identifier without further investigation. The review table should be handled at the review record level using the available key structure.

### **3.4.4 Foreign Keys Validation Results**

All tested foreign key relationships passed the validation with zero invalid references.

This confirms that the main transactional tables have consistent references to their corresponding parent tables. In particular, orders correctly reference customers, while order_items correctly reference orders, products, and sellers. Payments and reviews also correctly reference existing orders.

The validated relationships provide a reliable basis for integrating the datasets. However, joins must still consider the analytical grain of each table because valid relationships do not prevent row multiplication.

### **3.4.5 Analytical Grain Results and Summary**

#### **Summary**

| Table | Validated Analytical Grain |
|---|---|
| customers | One row per customer order record |
| orders | One row per order |
| order_items | One row per product item within an order |
| payments | One row per payment sequence within an order |
| reviews | One row per review record |
| products | One row per product |
| sellers | One row per seller |
| geolocation | One row per geolocation record |
| product_category_name_translation | One row per product category translation |

The analytical grain validation shows that the datasets operate at different levels of detail.

The customers table contains one record per customer order record. Although `customer_id` is unique across all 99,441 records, only 96,096 unique `customer_unique_id` values exist. This indicates that the same customer can appear in multiple customer records.

The orders table follows a one-row-per-order grain, with 99,441 unique `order_id` values and no duplicate order IDs.

The order_items table has a one-row-per-product-item-within-an-order grain. The combination of `order_id` and `order_item_id` is unique, while individual `order_id` values can occur multiple times because one order can contain multiple items.

The payments table follows a one-row-per-payment-sequence-within-an-order grain. The combination of `order_id` and `payment_sequential` is unique, while individual orders may contain multiple payment records.

The reviews table requires additional attention. The tested `review_id` contains 814 duplicate values, while the combination of `order_id` and `review_id` remains unique. Therefore, reviews should not be assumed to have a strictly unique `review_id` grain without further investigation.

The products and sellers tables each follow a one-row-per-product and one-row-per-seller grain, respectively.

The geolocation table contains 1,000,163 records but only 19,015 unique zip code prefixes. It also contains 261,831 exact duplicate rows. Therefore, its raw grain is one geolocation record rather than one row per zip code. A separate aggregated mapping can be created when one geographic record per zip code is required for analysis.

These findings confirm that the tables should not be joined indiscriminately. Tables with multiple records per order, such as order_items and payments, should be aggregated to the required analytical grain before being combined with order-level data. This approach will help prevent row multiplication and double counting in downstream analysis and dashboard metrics.